# PREPARE ENVIRONMENT

In [1]:
import os
import sys
import yaml
from pathlib import Path

# Add the parent directory (where "modules" is located) to the Python path
notebook_dir = os.getcwd()
parent_dir = os.path.dirname(notebook_dir)
sys.path.append(parent_dir)

# import functions from modules
from modules import llm_database_iteration_functions as ldif
from modules import llm_iteration_functions as lif

# get config data
conf = yaml.safe_load(Path(os.path.join(parent_dir, "config.yaml")).read_text(encoding="utf-8"))

# MODEL PULLING

Pull or download the `llama3.2:1b` model from the Ollama library.

In [2]:
import ollama

# download model to iterate with
ollama.pull("llama3.2:1b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

# SQL - RAG SIMULATION 2

Several steps chain are defined to process the user's query and retrieve relevant recipe suggestions:

- **Data Extraction**: The LLM, guided by predefined prompt behavior and template instructions, extracts key information from the user's query. It identifies and categorizes relevant data, such as recipe names, food names, ingredients, and ingredient categories. The LLM’s behavior template processes the query to create a structured JSON object with this information (e.g., recipe name, food name, ingredients, categories), marking the query as asking for a specific recipe or food as needed.

- **JSON Processing**: The system processes the JSON to determine necessary filters for querying the local database.

- **SQL Query Generation**: The `write_query` function is used to generate the SQL query based on the extracted data.

- **Query Execution**: The generated SQL query is executed using the `execute_query` function.

- **Recipe Suggestions**: Recipe suggestions are fetched and returned from the local database based on the user’s input, following the chain: `chain = write_query | execute_query`.

In [ ]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.llms import Ollama

# Load Ollama model
llm = Ollama(model="llama3.2:1b", temperature=0)  # Always same result for the same input

# get database_path
data_folder = os.path.join(parent_dir, conf["DATA_PATH"])
database_path = os.path.join(data_folder, conf["DATABASE_NAME"])

# Create an analysis prompt template
prompt_analysis = PromptTemplate(
    input_variables=["user_input"],
    template= lif.get_analysis_prompt_behavior()
)

# Create LLMChain for analysis
analysis_chain = LLMChain(llm=llm, prompt=prompt_analysis)

# Test the system with an example input
query1 = "Nire gisara pizza bat egin nahiko nuke. Eman ideiak."
print("---\n<USER>:", query1)


# Run the chain with just the user input
response1 = analysis_chain.run(user_input=lif.translate_text_with_Elia(query1, "en", "eu"))

# handle result
if response1:
    # Extract json from response1
    json_dict = lif.extract_json_as_dict_from_response(response1)
   
    if json_dict:
        # preprocess: obtain sql instructions and obtain informatian from ddbb
        ldif.process_json_dict_and_get_bbdd_result(json_dict, database_path , llm, "eu")

---
<USER>: Nire gisara pizza bat egin nahiko nuke. Eman ideiak.


In [4]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.llms import Ollama

# Load Ollama model
llm = Ollama(model="llama3.2:1b", temperature=0)  # Always same result for the same input

# get database_path
data_folder = os.path.join(parent_dir, conf["DATA_PATH"])
database_path = os.path.join(data_folder, conf["DATABASE_NAME"])

# Create an analysis prompt template
prompt_analysis = PromptTemplate(
    input_variables=["user_input"],
    template= lif.get_analysis_prompt_behavior()
)

# Create LLMChain for analysis
analysis_chain = LLMChain(llm=llm, prompt=prompt_analysis)

# Test the system with an example input
recipe = "zurrukutuna"
query2 = f"{recipe} prestatu nahiko nuke. Nola egiten da?"
print("---\n<USER>:", query2)

# Run the chain with just the user input
response2 = analysis_chain.run(user_input=query2)

# handle result
if response2:
    # Extract json from response2
    json_dict = lif.extract_json_as_dict_from_response(response2)
   
    if json_dict:
        # preprocess: obtain sql instructions and obtain informatian from ddbb
        ldif.process_json_dict_and_get_bbdd_result(json_dict, database_path , llm, "eu")

---
<USER>: zurrukutuna prestatu nahiko nuke. Nola egiten da?
---
<MODEL>: Ez daukat errezeta zehatzik "zurrukutuna"rekin lotuta, baina errezeta horrek ['Bakailao gazia', 'baratxuri', 'ogi', 'oliba-olio'] osagaiak erabiltzen ditu; beraz, osagai horiekin egin daitezkeen 5 errezeta iradokitzen saiatuko naiz (asko jota).
-1. proposamena: Antxoa baratxuritan
Prestatzeko argibideak hemen daude: https://eu.wikibooks.org/wiki/Sukaldaritza_liburua/Errezetak/Antxoa_baratxuritan_frijitua.
-2. proposamena: Ganbak baratxuritan
Prestatzeko argibideak hemen daude: https://eu.wikibooks.org/wiki/Sukaldaritza_liburua/Errezetak/Ganbak_baratxuritan.


The result effectively suggests recipes related to `zurrukutuna` parts of ingredients, but it could be further refined to offer a wider variety of cod-based dishes by fine-tuning the ingredient matching, enabling a broader selection of relevant recipes and variations.